In [2]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, save, output_file
from bokeh.layouts import column, row
from bokeh.models import HoverTool, Span, Label, LinearColorMapper, ColorBar, ColumnDataSource
from bokeh.palettes import RdYlBu11, Spectral6
from bokeh.transform import transform
from datetime import datetime, timedelta
import random

# Crear datos financieros simulados realistas
np.random.seed(42)
fechas = pd.date_range(start='2020-01-01', end='2024-12-31', freq='D')
n_dias = len(fechas)

# Simular precios de múltiples acciones
acciones = {
    'AAPL': {'precio_inicial': 75, 'volatilidad': 0.02, 'tendencia': 0.0003},
    'MSFT': {'precio_inicial': 160, 'volatilidad': 0.018, 'tendencia': 0.0004},
    'GOOGL': {'precio_inicial': 1400, 'volatilidad': 0.025, 'tendencia': 0.0002},
    'TSLA': {'precio_inicial': 400, 'volatilidad': 0.04, 'tendencia': 0.0005},
    'AMZN': {'precio_inicial': 1800, 'volatilidad': 0.022, 'tendencia': 0.0001}
}

datos_mercado = []
for simbolo, config in acciones.items():
    precios = [config['precio_inicial']]
    volumenes = []
    
    for i in range(1, n_dias):
        # Movimiento browniano geométrico para precios realistas
        cambio = np.random.normal(config['tendencia'], config['volatilidad'])
        nuevo_precio = precios[-1] * (1 + cambio)
        precios.append(max(nuevo_precio, 1))  # Evitar precios negativos
        
        # Volumen correlacionado inversamente con el precio
        volumen_base = 1000000
        volatilidad_precio = abs(cambio)
        volumen = int(volumen_base * (1 + volatilidad_precio * 10) * random.uniform(0.5, 2))
        volumenes.append(volumen)
    
    volumenes.insert(0, 1000000)  # Volumen inicial
    
    for i, fecha in enumerate(fechas):
        datos_mercado.append({
            'fecha': fecha,
            'simbolo': simbolo,
            'precio': precios[i],
            'volumen': volumenes[i],
            'precio_anterior': precios[i-1] if i > 0 else precios[i],
            'cambio_pct': ((precios[i] - precios[i-1]) / precios[i-1] * 100) if i > 0 else 0
        })

df_mercado = pd.DataFrame(datos_mercado)

# GRÁFICO 1: Líneas de tiempo múltiples con interactividad avanzada
def crear_grafico_precios():
    # Configurar hover tool ANTES de crear la figura
    hover = HoverTool(tooltips=[
        ("Empresa", "@simbolo"),
        ("Fecha", "@fecha{%F}"),
        ("Precio", "$@precio{0.00}"),
        ("Cambio %", "@cambio_pct{0.00}%"),
        ("Volumen", "@volumen{0,0}")
    ], formatters={'@fecha': 'datetime'})
    
    p1 = figure(
        title="📈 Dashboard de Precios de Acciones Tecnológicas (2020-2024)",
        x_axis_type='datetime',
        width=1200,
        height=500,
        tools=[hover, "pan", "wheel_zoom", "box_zoom", "reset", "save"],
        active_scroll='wheel_zoom'
    )
    
    # Configuración profesional del diseño
    p1.title.text_font_size = "18pt"
    p1.title.text_color = "#2c3e50"
    p1.title.text_font = "Helvetica"
    p1.title.text_font_style = "bold"
    
    # Fondo y bordes elegantes
    p1.background_fill_color = "#fafafa"
    p1.border_fill_color = "#ffffff"
    p1.outline_line_color = "#cccccc"
    
    # Configurar ejes
    p1.xaxis.axis_label = "Período"
    p1.yaxis.axis_label = "Precio (USD)"
    p1.xaxis.axis_label_text_font_size = "14pt"
    p1.yaxis.axis_label_text_font_size = "14pt"
    p1.axis.axis_label_text_color = "#2c3e50"
    
    # Grid profesional
    p1.grid.grid_line_alpha = 0.3
    p1.grid.grid_line_color = "#cccccc"
    
    colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    
    # Añadir líneas para cada acción
    for i, simbolo in enumerate(acciones.keys()):
        data = df_mercado[df_mercado['simbolo'] == simbolo]
        
        # Crear ColumnDataSource para mejor interactividad
        source = ColumnDataSource(data)
        
        # Línea principal
        p1.line('fecha', 'precio', 
                source=source,
                line_width=3, 
                color=colores[i], 
                alpha=0.8,
                legend_label=simbolo)
        
        # CAMBIO: Usar scatter en lugar de circle para evitar warning
        p1.scatter('fecha', 'precio', 
                  source=source,
                  size=4, 
                  color=colores[i], 
                  alpha=0.6)
    
    # Configurar leyenda profesional
    p1.legend.location = "top_left"
    p1.legend.click_policy = "hide"  # Permite ocultar/mostrar líneas
    p1.legend.background_fill_alpha = 0.8
    p1.legend.border_line_color = "#cccccc"
    p1.legend.label_text_font_size = "12pt"
    
    # Añadir líneas de referencia importantes (eventos del mercado)
    covid_span = Span(location=pd.Timestamp('2020-03-15'), 
                     dimension='height', 
                     line_color='red', 
                     line_dash='dashed', 
                     line_alpha=0.7,
                     line_width=2)
    p1.add_layout(covid_span)
    
    covid_label = Label(x=pd.Timestamp('2020-03-15'), y=max(df_mercado['precio'])*0.9,
                       text="COVID-19 Crisis", 
                       text_color='red',
                       text_font_size='10pt')
    p1.add_layout(covid_label)
    
    return p1

# GRÁFICO 2: Mapa de calor de correlaciones
def crear_mapa_correlaciones():
    # Crear matriz de datos para correlación
    pivot_data = df_mercado.pivot(index='fecha', columns='simbolo', values='precio')
    correlaciones = pivot_data.corr()
    
    # Preparar datos para el mapa de calor
    simbolos = list(correlaciones.columns)
    datos_heatmap = []
    
    for i, sym1 in enumerate(simbolos):
        for j, sym2 in enumerate(simbolos):
            datos_heatmap.append({
                'x': sym1,
                'y': sym2, 
                'correlacion': correlaciones.loc[sym1, sym2],
                'color_val': correlaciones.loc[sym1, sym2]
            })
    
    df_heatmap = pd.DataFrame(datos_heatmap)
    source_heatmap = ColumnDataSource(df_heatmap)
    
    # Crear figura
    p2 = figure(
        title="🔥 Matriz de Correlación entre Acciones",
        x_range=simbolos,
        y_range=list(reversed(simbolos)),
        width=600,
        height=500,
        tools="hover,save",
        toolbar_location='above'
    )
    
    # Configurar color mapper
    mapper = LinearColorMapper(palette=RdYlBu11, low=-1, high=1)
    
    # Crear rectángulos del mapa de calor
    p2.rect(x='x', y='y', width=1, height=1,
           source=source_heatmap,
           fill_color=transform('color_val', mapper),
           line_color=None)
    
    # Configurar hover
    p2.hover.tooltips = [
        ("Par", "@x - @y"),
        ("Correlación", "@correlacion{0.000}")
    ]
    
    # Añadir barra de colores
    color_bar = ColorBar(color_mapper=mapper, width=8, location=(0,0))
    p2.add_layout(color_bar, 'right')
    
    # Estilización profesional
    p2.title.text_font_size = "16pt"
    p2.title.text_color = "#2c3e50"
    p2.axis.axis_label_text_font_size = "12pt"
    p2.axis.major_label_text_font_size = "10pt"
    p2.grid.grid_line_color = None
    
    return p2

# GRÁFICO 3: Análisis de Volumen vs Precio
def crear_scatter_volumen():
    # Preparar datos agregados por mes
    df_mercado['año_mes'] = df_mercado['fecha'].dt.to_period('M')
    datos_agregados = df_mercado.groupby(['simbolo', 'año_mes']).agg({
        'precio': 'mean',
        'volumen': 'mean',
        'cambio_pct': 'std'
    }).reset_index()
    
    p3 = figure(
        title="💹 Análisis: Volumen vs Precio Promedio Mensual",
        width=800,
        height=600,
        tools="pan,wheel_zoom,box_select,reset,save"
    )
    
    # Configurar ejes
    p3.xaxis.axis_label = "Volumen Promedio"
    p3.yaxis.axis_label = "Precio Promedio (USD)"
    
    colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    
    for i, simbolo in enumerate(acciones.keys()):
        data = datos_agregados[datos_agregados['simbolo'] == simbolo]
        
        # CAMBIO: Ya usaba scatter correctamente, no necesita cambio
        p3.scatter(data['volumen'], data['precio'],
                  size=data['cambio_pct']*3,  # Tamaño basado en volatilidad
                  color=colores[i],
                  alpha=0.7,
                  legend_label=simbolo)
    
    p3.legend.location = "top_right"
    p3.title.text_font_size = "16pt"
    p3.title.text_color = "#2c3e50"
    
    return p3

# Crear todos los gráficos
grafico_precios = crear_grafico_precios()
mapa_correlaciones = crear_mapa_correlaciones()
scatter_volumen = crear_scatter_volumen()

# Crear layout profesional
dashboard = column(
    grafico_precios,
    row(mapa_correlaciones, scatter_volumen)
)

# Guardar archivo
output_file("../GraficosProfecionales/html/dashboard_financiero.html")
save(dashboard)

print("✅ Dashboard Financiero Profesional creado: GraficosProfecionales/html/dashboard_financiero.html")
print("🎯 Dashboard listo para portafolio profesional!")

✅ Dashboard Financiero Profesional creado: GraficosProfecionales/html/dashboard_financiero.html
🎯 Dashboard listo para portafolio profesional!
